# Imports

In [1]:
import threading, time, json, re
import pandas as pd
from collections import deque
from queue import Queue, Empty
from openai import OpenAI
from tqdm.auto import tqdm

# Configuration

In [2]:
API_KEY    = "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"
MODEL      = "qwen/qwen3-next-80b-a3b-thinking"
BASE_URL   = "https://integrate.api.nvidia.com/v1"
INPUT_CSV  = "/home/mabdrabou/Desktop/NLP Project/mental_health.csv"
OUTPUT_CSV = "/home/mabdrabou/Desktop/NLP Project/qwen_labels_output.csv"

In [3]:
RPM_LIMIT  = 28
WINDOW_SEC = 60
REST_SEC   = 35

# Few Shot Prompt + LLM For Labeling

In [4]:
SUMMARY_PROMPT = """\
# Instruction
You are a clinical NLP expert and mental health classifier. Your task is to carefully read a person's text message and assess their mental state on a scale of 1 to 10.

Here is the brief description about the dataset:
To classify if the person is struggling with mental health issues or not from their messages.
I will provide you with the user text and a label. Label with 0 means mentally well and score will usually be less than or equal to 5. Label with 1 means not mentally well, which means score would be greater than 5.
This dataset is a collection of texts related to people with anxiety, depression, and other mental health issues. The corpus consists of two columns: one containing the comments, and the other containing labels indicating whether the comments are considered indicative of mental health issues or not. The data may be useful for researchers, mental health professionals, and others interested in understanding the language and sentiment surrounding mental health issues. 1 means considered as a comment which is indicative of mental health issues, and 0 means a person who is not struggling with mental health issues.

Understanding and detecting mental health issues through text messages can be a critical step in providing timely support and intervention for those in need. Research has shown that linguistic patterns and word choices in written communication can be indicative of various mental health conditions, including depression, anxiety, and stress. Analyzing the content of messages, along with the intensity of emotions conveyed, can offer valuable insights into a person's emotional well-being.

This cutting-edge field combines NLP techniques with psychology and psychiatry, aiming to build automated systems capable of identifying signs of mental distress accurately.

You should first read the user input carefully, then evaluate based on the Criteria and Rating Rubric below. Give step-by-step reasoning for your rating.

---

# Evaluation

## Metric Definition
You will be assessing the mental health state of a person based on their text. The response must be grounded only in the provided text — do not infer or assume information not present.

## Criteria
- **Mental Wellbeing**: Overall emotional and psychological state of the person.
- **Hopelessness**: Presence of nihilistic thinking, loss of purpose, or belief that things cannot improve.
- **Anxiety**: Signs of excessive worry, fear, panic, or rumination.
- **Depression**: Signs of persistent sadness, withdrawal, low energy, or emotional numbness.
- **Stress**: Indicators of being overwhelmed, burned out, or under pressure.
- **Suicidal Ideation**: Any direct or indirect references to self-harm or not wanting to live.
- **Groundedness**: Assessment must be based only on what the text says; do not hallucinate signals.

## Rating Rubric
- 1 (BEST): Excellent mental state. No signs of distress or mental health concerns whatsoever.
- 2 (VERY_GOOD): Mentally well with very minor or no issues; no immediate concerns.
- 3 (GOOD): Healthy mental state with minor or occasional concerns; generally doing well.
- 4 (MODERATE_GOOD): Fairly good mental state; minimal issues present.
- 5 (NEUTRAL): Borderline — neither clearly healthy nor unhealthy; some ambiguous signals.
- 6 (MODERATE_BAD): Moderate signs of mental health issues; monitoring or professional support advised.
- 7 (BAD): Significantly compromised mental state; recommend seeking help soon.
- 8 (VERY_BAD): Severely compromised; professional assistance needed promptly.
- 9 (VERY_VERY_BAD): Very severely compromised; immediate intervention strongly advised.
- 10 (WORST): Extremely poor mental state; urgent crisis intervention required.

## Evaluation Steps
- STEP 1: Read the text carefully and identify any linguistic signals related to mental health (hopelessness, anxiety, depression, suicidal ideation, stress, isolation, etc.).
- STEP 2: Cross-reference identified signals against the Criteria above.
- STEP 3: Assign a score strictly following the Rating Rubric, ensuring consistency with the provided label (label 0 → score ≤ 5, label 1 → score > 5).
- STEP 4: List the dominant signals observed in the text.
- STEP 5: Write a concise 2–3 sentence reasoning explaining your score.

---

# Few-Shot Examples

## Score 1 — BEST
**Text:** "Had the most amazing weekend hiking with friends. Feeling so refreshed and grateful for life. Can't wait for next weekend!"
**Label:** 0 (mentally well)
**Reasoning:** Expresses joy, gratitude, and social connection. No indicators of distress, anxiety, or depression. Person is thriving.
**Dominant Signals:** positive affect, social engagement, forward-looking mindset
**Score:** 1
**Rating:** BEST

---

## Score 2 — VERY_GOOD
**Text:** "guys finally got a girlfriend after leaving a toxic relationship that had a negative effect on my wellbeing. got help writing a text, my dad said he was proud of how i dealt with it. could barely believe it. nice to see there are loads of posts about people getting into good relationships"
**Label:** 0 (mentally well)
**Reasoning:** Positive and forward-looking narrative. Healthy coping and growth after recovering from a difficult relationship. Strong support network present.
**Dominant Signals:** recovery, positive affect, social support, healthy growth
**Score:** 2
**Rating:** VERY_GOOD

---

## Score 3 — GOOD
**Text:** "Work has been a bit stressful lately but I've been managing it fine. Going to the gym helps a lot. My friends have been super supportive too."
**Label:** 0 (mentally well)
**Reasoning:** Acknowledges some stress but demonstrates active and healthy coping mechanisms. Strong social and physical health buffers are in place.
**Dominant Signals:** mild work stress, healthy coping, social support, physical activity
**Score:** 3
**Rating:** GOOD

---

## Score 4 — MODERATE_GOOD
**Text:** "tell my crush i like her ive been procrastinating for months at this point im still unsure about it get help pls"
**Label:** 0 (mentally well)
**Reasoning:** Person is nervous about a normal social situation. Shows some anxiety around interpersonal interaction but nothing indicative of a mental health disorder. Seeking lighthearted advice.
**Dominant Signals:** social nervousness, mild indecision, no clinical distress signals
**Score:** 4
**Rating:** MODERATE_GOOD

---

## Score 5 — NEUTRAL
**Text:** "I don't really know how I'm feeling these days. Some days are okay, some days just feel really heavy. I'm managing but it's not easy."
**Label:** 0 (mentally well)
**Reasoning:** Ambiguous emotional state — neither clearly distressed nor clearly healthy. The person is coping but borderline; emotional heaviness is noted without acute crisis signals.
**Dominant Signals:** emotional ambiguity, fluctuating mood, mild low affect, no acute crisis
**Score:** 5
**Rating:** NEUTRAL

---

## Score 6 — MODERATE_BAD
**Text:** "world ppl cares give them i planet yrs one thing learned ppl care u something give them im tired wish born way care productive wanna connect ppl want cant connect anyone awful feel trapped"
**Label:** 1 (at risk)
**Reasoning:** Deep social isolation, feeling trapped, exhaustion, and inability to connect with others are present. Multiple moderate depression markers including emotional withdrawal and hopelessness are evident.
**Dominant Signals:** social isolation, feeling trapped, hopelessness, emotional withdrawal, fatigue
**Score:** 6
**Rating:** MODERATE_BAD

---

## Score 7 — BAD
**Text:** "i've been crying every day for weeks and i don't even know why. i can't get out of bed most mornings. i've stopped replying to my friends. everything just feels pointless and heavy."
**Label:** 1 (at risk)
**Reasoning:** Persistent depressive symptoms including anhedonia, social withdrawal, unexplained crying, and loss of motivation are all present. Functioning is clearly impaired and professional help is recommended.
**Dominant Signals:** persistent crying, social withdrawal, anhedonia, impaired functioning, low motivation
**Score:** 7
**Rating:** BAD

---

## Score 8 — VERY_BAD
**Text:** "dont know how long i can hold on longer. thought my boyfriend could be enough to help hold me. possibly going to start meds even though i hate anything like that. theres nothing going anywhere in my life. its fucking meaningless. cant continue anymore."
**Label:** 1 (at risk)
**Reasoning:** Clear hopelessness, perceived meaninglessness, and strong implicit suicidal ideation are present. Person feels unsupported and is struggling severely with daily functioning.
**Dominant Signals:** hopelessness, meaninglessness, implicit suicidal ideation, emotional exhaustion, loss of will
**Score:** 8
**Rating:** VERY_BAD

---

## Score 9 — VERY_VERY_BAD
**Text:** "nothing to look forward to in life. dont have many reasons to keep going. feel like nothing keeps me going to the next day. makes me want to hang."
**Label:** 1 (at risk)
**Reasoning:** Directly expresses suicidal ideation and a complete absence of hope or reason to live. Crisis-level message requiring immediate intervention.
**Dominant Signals:** explicit suicidal ideation, total hopelessness, no future orientation, desire to die
**Score:** 9
**Rating:** VERY_VERY_BAD

---

## Score 10 — WORST
**Text:** "cant do this anymore. tried to kill myself twice. wish id succeeded last summer, a few months ago. was hospitalized. couldnt do anything. cant say it really helped. told my pdoc and therapist as well two weeks ago. not sure about telling people anymore."
**Label:** 1 (at risk)
**Reasoning:** Two prior suicide attempts with expressed regret at survival, recent hospitalization, and eroding trust in professional help. This is an extreme, immediate crisis situation.
**Dominant Signals:** multiple suicide attempts, survivor's regret, loss of trust in professionals, active crisis, isolation from support
**Score:** 10
**Rating:** WORST

---

# User Input

## Text Response
{text}

## Label
{label}

---

Respond with ONLY this JSON and nothing else — no markdown, no explanation, no preamble:
{{"reasoning": "2-3 sentences", "dominant_signals": ["signal1", "signal2", "signal3"], "score": <1-10>, "rating": "<RATING>"}}
"""

def build_prompt(text, label):
    return SUMMARY_PROMPT.format(
        text=str(text),
        label=f"{label} ({'mentally well' if label == 0 else 'at risk'})"
    )


def extract_json(raw: str):
    """Strip think-blocks / fences, then parse the first valid JSON object."""
    raw = re.sub(r"<think>.*?</think>", "", raw, flags=re.DOTALL)
    raw = re.sub(r"```(?:json)?", "", raw).strip()

    required = {"reasoning", "dominant_signals", "score", "rating"}

    for match in re.finditer(r"\{[^{}]*\}", raw, re.DOTALL):
        try:
            data = json.loads(match.group())
            if required.issubset(data):
                data["score"] = int(data["score"])
                return data
        except (json.JSONDecodeError, ValueError):
            continue

    try:
        s, e = raw.rfind("{"), raw.rfind("}") + 1
        if s != -1 and e > s:
            data = json.loads(raw[s:e])
            data["score"] = int(data["score"])
            return data
    except Exception:
        pass

    return None


_request_times: list[float] = []

def rate_limited_wait():
    while True:
        now = time.time()
        while _request_times and now - _request_times[0] >= WINDOW_SEC:
            _request_times.pop(0)

        if len(_request_times) < RPM_LIMIT:
            _request_times.append(now)
            return

        sleep_for = max(REST_SEC, WINDOW_SEC - (now - _request_times[0]) + 1)
        print(f"\n  [rate-limit] sleeping {sleep_for:.0f}s …")
        time.sleep(sleep_for)


client = OpenAI(base_url=BASE_URL, api_key=API_KEY)

def label_row(text: str, label: int, max_retries: int = 5):
    """Call the API (sequential, single key) and return parsed JSON or None."""
    backoff = REST_SEC
    prompt  = build_prompt(text, label)

    for attempt in range(1, max_retries + 1):
        try:
            rate_limited_wait()

            stream = client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.6,
                top_p=0.7,
                max_tokens=10_000,
                stream=True,
            )

            raw = ""
            for chunk in stream:
                delta = chunk.choices[0].delta if chunk.choices else None
                if delta and delta.content:
                    raw += delta.content

            result = extract_json(raw)
            if result:
                return result

            print(f"\n  [row] attempt {attempt}: JSON parse failed, retrying…")

        except Exception as exc:
            msg = str(exc)
            print(f"\n  [row] attempt {attempt} error: {msg[:120]}")
            if "429" in msg:
                time.sleep(backoff)
                backoff = min(backoff * 2, 120)

    return None


def label_dataframe(df: pd.DataFrame,
                    text_col: str = "text",
                    label_col: str = "label") -> pd.DataFrame:

    total = len(df)
    print(f"Labeling {total:,} rows sequentially with one API key …\n")

    reasoning_col   = []
    score_col       = []
    rating_col      = []
    signals_col     = []

    failed_indices  = []

    with tqdm(total=total, unit="row", colour="cyan") as pbar:
        for idx, row in df.iterrows():
            result = label_row(str(row[text_col]), int(row[label_col]))

            if result:
                reasoning_col.append(result["reasoning"])
                score_col.append(result["score"])
                rating_col.append(result["rating"])
                signals_col.append(", ".join(result["dominant_signals"]))
            else:
                reasoning_col.append(None)
                score_col.append(None)
                rating_col.append(None)
                signals_col.append(None)
                failed_indices.append(idx)

            pbar.update(1)

    print(f"\nPass 1 done — labeled: {total - len(failed_indices):,}  |  failed: {len(failed_indices):,}")

    if failed_indices:
        print(f"\nRetrying {len(failed_indices)} failed rows …")
        still_failed = []

        with tqdm(total=len(failed_indices), unit="row", colour="yellow") as pbar:
            for idx in failed_indices:
                pos = df.index.get_loc(idx)        # positional index in df
                result = label_row(str(df.loc[idx, text_col]),
                                   int(df.loc[idx, label_col]))
                if result:
                    reasoning_col[pos]  = result["reasoning"]
                    score_col[pos]      = result["score"]
                    rating_col[pos]     = result["rating"]
                    signals_col[pos]    = ", ".join(result["dominant_signals"])
                else:
                    still_failed.append(idx)
                pbar.update(1)

        print(f"Retry done — recovered: {len(failed_indices) - len(still_failed):,}  "
              f"|  still missing: {len(still_failed):,}")

    out = df.copy()
    out["qwen_reasoning"]        = reasoning_col
    out["qwen_score"]            = score_col
    out["qwen_rating"]           = rating_col
    out["qwen_dominant_signals"] = signals_col

    out.to_csv(OUTPUT_CSV, index=False)
    labeled = out["qwen_score"].notna().sum()
    print(f"\nSaved → {OUTPUT_CSV}  ({labeled:,}/{total:,} labeled)")
    return out

In [ ]:
if __name__ == "__main__":
    df = pd.read_csv(INPUT_CSV)
    df_labeled = label_dataframe(df, text_col="text", label_col="label")
    print(df_labeled.head())